# RecVis Assignment 3 — CIFAR-10 → synthetic test set

Three ways to classify the test images, each one image at a time; every run is one experiment:

| Section | Method | Options |
|---|---|---|
| 1 | ResNet / ViT (torchvision or timm) | from scratch, linear probe, partial or full fine-tuning, augmentations |
| 2 | CLIP zero-shot | `prompt_set`: `base`, `cifar`, `imagenet`, `cifar+imagenet` |
| 2 | CLIP linear probe + WiSE-FT | `wise_alpha` (0.25, 0.5, 0.75, 1.0), head initialized from the zero-shot classifier |

Run **Setup**, then one section. Every run writes a new folder `results/<YYYYmmdd_HHMM>/`:
`config.yaml`, `run.log`, `submission_*.csv`, `logits/` (test and validation logits), `cache/` (features: test always,
validation of the selected model, train only for linear probes), `checkpoints/`, training histories.
`features_from=<experiment folder>` reuses the features of its `cache/` instead of recomputing them.

In [ ]:
# Comment this line out if the packages are already installed
%pip install -q open-clip-torch timm pyyaml

## Setup

### Imports

In [1]:
import dataclasses
import logging
import math
import os
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import open_clip
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.models as models
import torchvision.transforms as T
import yaml
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm

### Configuration

In [ ]:
os.environ["RECVIS_INPUT_DIR"] = str(Path.cwd() / "data")
os.environ["RECVIS_OUTPUT_DIR"] = str(Path.cwd() / "results")

In [ ]:
@dataclass
class GlobalConfig:
    input_dir: Path = Path(os.environ.get('RECVIS_INPUT_DIR', '/kaggle/input/recvis-mva-a3-2026'))
    results_dir: Path = Path(os.environ.get('RECVIS_OUTPUT_DIR', '/kaggle/working/results'))
    seed: int = 42
    class_names: tuple = ('airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
    cifar_mean: tuple = (0.4914, 0.4822, 0.4465)
    cifar_std: tuple = (0.2470, 0.2435, 0.2616)
    num_workers: int = min(4, os.cpu_count() or 2)

    @property
    def num_classes(self):
        return len(self.class_names)

CFG = GlobalConfig()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
AMP = dict(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == 'cuda')
USE_GRAD_SCALER = AMP['enabled'] and AMP_DTYPE == torch.float16

### Run

In [4]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    g = torch.Generator()
    g.manual_seed(seed)
    worker_init_fn = seed_worker

    return g, worker_init_fn

def safe_tag(*parts):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', '_'.join(str(p) for p in parts))

def to_plain(obj):
    """Configs and results -> YAML-friendly Python objects"""
    if dataclasses.is_dataclass(obj) and not isinstance(obj, type):
        return {f.name: to_plain(getattr(obj, f.name)) for f in dataclasses.fields(obj)}
    if isinstance(obj, dict):
        return {str(k): to_plain(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_plain(v) for v in obj]
    if isinstance(obj, (np.ndarray, torch.Tensor)):
        return np.asarray(obj).tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, str):
        return str(obj)
    if obj is None or isinstance(obj, (bool, int, float)):
        return obj

    return str(obj)

def environment_info(device, amp):
    info = {
        'device': str(device),
        'autocast': str(amp['dtype']) if amp['enabled'] else 'float32',
        'torch': torch.__version__,
        'torchvision': torchvision.__version__,
        'timm': timm.__version__,
        'open_clip': open_clip.__version__,
    }

    if device.type == 'cuda':
        info['gpu'] = torch.cuda.get_device_name(device)

    return info

class Run:
    """Folder results/<YYYYmmdd_HHMM>/ with config.yaml, run.log, submission_*.csv, logits/, checkpoints/, cache/, *.csv"""

    def __init__(self, section, config, global_config, environment):
        self.section, self.config = section, config
        self.global_config, self.environment = global_config, environment
        self.started = datetime.now()
        self.dir = self.new_folder()

        for sub in ('logits', 'checkpoints', 'cache'):
            (self.dir / sub).mkdir()

        self.results, self.files, self.metrics = {}, [], []
        root = logging.getLogger()

        for handler in [h for h in root.handlers if isinstance(h, logging.FileHandler)]:
            root.removeHandler(handler)
            handler.close()

        handler = logging.FileHandler(self.dir / 'run.log', encoding='utf-8')
        handler.setFormatter(root.handlers[0].formatter)
        root.addHandler(handler)
        self.generator, self.worker_init_fn = set_seed(config.seed)
        self.save_config()
        log.info('run folder %s | %s', self.dir, to_plain(config))

    def new_folder(self):
        results_dir = self.global_config.results_dir
        results_dir.mkdir(parents=True, exist_ok=True)
        stamp = self.started.strftime('%Y%m%d_%H%M')

        for name in [stamp] + [f"{self.started.strftime('%Y%m%d_%H%M%S')}_{i}" for i in range(1, 100)]:
            try:
                (results_dir / name).mkdir()
                return results_dir / name
            except FileExistsError:
                continue

        raise RuntimeError(f'No free run folder in {results_dir}')

    def path(self, *parts):
        path = self.dir.joinpath(*parts)
        relative = str(path.relative_to(self.dir))

        if relative not in self.files:
            self.files.append(relative)

        return path

    def save_config(self):
        content = to_plain(
            {
                'section': self.section,
                'started': self.started.isoformat(timespec='seconds'),
                'environment': self.environment,
                'global_config': self.global_config,
                'section_config': self.config,
                'results': self.results,
                'files': self.files,
            }
        )
        text = yaml.safe_dump(content, sort_keys=False, allow_unicode=True, width=120)
        (self.dir / 'config.yaml').write_text(text, encoding='utf-8')

    def record(self, key, value):
        self.results[key] = to_plain(value)
        self.save_config()

    def save_features(self, name, features):
        np.save(self.path('cache', f'{name}.npy'), torch.as_tensor(features).numpy().astype(np.float16))

    def save_submission(self, name, test_logits, val_logits, val_labels, sample_submission):
        """Writes submission_<name>.csv, logits/<name>.npy (test), logits/<name>_validation.npy and a row of metrics.csv"""
        test_logits, val_logits = torch.as_tensor(test_logits).float(), torch.as_tensor(val_logits).float()
        predictions = test_logits.argmax(1).numpy()

        if len(predictions) != len(sample_submission):
            raise ValueError(f'Got {len(predictions)} predictions for {len(sample_submission)} test images')

        sample_submission.assign(label=predictions).to_csv(self.path(f'submission_{name}.csv'), index=False)
        np.save(self.path('logits', f'{name}.npy'), test_logits.numpy())
        np.save(self.path('logits', f'{name}_validation.npy'), val_logits.numpy())

        metrics = {
            'name': name,
            'cifar_val_acc': accuracy(val_logits, val_labels),
            'val_mean_max_prob': round(val_logits.softmax(1).max(1).values.mean().item(), 4),
            'test_mean_max_prob': round(test_logits.softmax(1).max(1).values.mean().item(), 4),
            'test_class_shares': (np.bincount(predictions, minlength=test_logits.shape[1]) / len(predictions)).round(4),
        }
        self.metrics.append(metrics)
        self.save_table('metrics', [{k: v for k, v in m.items() if k != 'test_class_shares'} for m in self.metrics])
        self.record(f'submission_{name}', metrics)
        log.info('saved submission_%s.csv | %s', name, metrics)

    def save_table(self, name, table):
        pd.DataFrame(table).to_csv(self.path(f'{name}.csv'), index=False)
        self.save_config()

    def save_checkpoint(self, name, obj):
        torch.save(obj, self.path('checkpoints', f'{name}.pth'))
        self.save_config()

    def finish(self):
        self.record('duration_seconds', round((datetime.now() - self.started).total_seconds(), 1))
        log.info('run finished: %s (%d files)', self.dir, len(self.files))

### Utilities

In [ ]:
def load_npz(path):
    with np.load(path) as archive:
        images = archive['images'].astype(np.uint8)
        labels = archive['labels'].astype(np.int64) if 'labels' in archive.files else None
        ids = archive['ids'].astype(np.int64) if 'ids' in archive.files else np.arange(len(images))

    return images, labels, ids

def to_model_input(batch, size, mean, std, interpolation, device):
    """Uint8 batch (NHWC or NCHW) -> normalized float NCHW on device, resized there"""
    x = batch.to(device, non_blocking=True)

    if x.shape[1] != 3:
        x = x.permute(0, 3, 1, 2)

    x = x.float().div_(255)

    if x.shape[-1] != size:
        align = False if interpolation in ('bilinear', 'bicubic') else None
        x = F.interpolate(x, size=(size, size), mode=interpolation, align_corners=align).clamp_(0, 1)

    mean = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    std = torch.tensor(std, device=device).view(1, 3, 1, 1)

    return (x - mean) / std

@torch.no_grad()
def extract_features(
    feature_fn, images, *, size, mean, std, interpolation, tta_flip, normalize, device, amp, batch_size=256, desc='features'
):
    """Features of uint8 NHWC images (CPU float32); with tta_flip, mean of the image and its horizontal flip"""
    images = torch.from_numpy(images)
    outputs = []

    for first in tqdm(range(0, len(images), batch_size), desc=desc, unit='batch', dynamic_ncols=True):
        batch = images[first : first + batch_size]
        views = []

        for view in ([batch, batch.flip(2)] if tta_flip else [batch]):
            with torch.autocast(**amp):
                features = feature_fn(to_model_input(view, size, mean, std, interpolation, device)).float()

            views.append(F.normalize(features, dim=-1) if normalize else features)

        features = torch.stack(views).mean(0)
        outputs.append((F.normalize(features, dim=-1) if normalize else features).cpu())

    return torch.cat(outputs)

@torch.no_grad()
def linear_logits(features, weight, bias=None, *, device, batch_size=8192):
    weight = torch.as_tensor(weight, dtype=torch.float32, device=device)
    outputs = []

    for chunk in torch.as_tensor(features).split(batch_size):
        logits = chunk.to(device, dtype=torch.float32) @ weight.T
        outputs.append((logits if bias is None else logits + torch.as_tensor(bias, device=device)).cpu())

    return torch.cat(outputs)

def load_features(folder, name, rows):
    """Features <folder>/cache/<name>.npy of a previous experiment, or None (folder None or file missing)"""
    if folder is None:
        return None

    path = Path(folder) / 'cache' / f'{name}.npy'

    if not path.is_file():
        log.warning('Features %s not found: computing them', path)
        return None

    features = torch.from_numpy(np.load(path).astype(np.float32))

    if len(features) != rows:
        raise ValueError(f'File {path} has {len(features)} rows, expected {rows}')

    log.info('loaded %s %s', path, tuple(features.shape))

    return features

def accuracy(logits, labels):
    return float((torch.as_tensor(logits).argmax(1) == torch.as_tensor(labels)).float().mean())

def make_optimizer(groups, name, *, lr, weight_decay, momentum=0.9, betas=(0.9, 0.999)):
    """Groups: list of {'params', 'lr', 'name'} (one LR per group)"""
    if name == 'adamw':
        return torch.optim.AdamW(groups, lr=lr, weight_decay=weight_decay, betas=tuple(betas))
    if name == 'adam':
        return torch.optim.Adam(groups, lr=lr, weight_decay=weight_decay, betas=tuple(betas))
    if name == 'sgd':
        return torch.optim.SGD(groups, lr=lr, weight_decay=weight_decay, momentum=momentum)

    raise ValueError(f'Unknown optimizer {name}')

def make_scheduler(optimizer, name, total_steps, warmup_fraction=0.0):
    """Stepped per batch; linear warmup over warmup_fraction of the steps, then cosine or constant

    OneCycle uses warmup_fraction as its own warmup phase; every scheduler keeps the LR of each parameter group
    """
    total_steps = max(1, total_steps)

    if name == 'onecycle':
        max_lr = [group['lr'] for group in optimizer.param_groups]
        return torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=max_lr, total_steps=total_steps, pct_start=max(warmup_fraction, 1 / total_steps)
        )

    if name not in ('cosine', 'constant'):
        raise ValueError(f'Unknown scheduler {name}')

    warmup_steps = round(warmup_fraction * total_steps)

    def lr_factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        if name == 'constant':
            return 1.0

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)

        return 0.5 * (1 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)

def train_head(
    train_features,
    train_labels,
    val_features,
    val_labels,
    *,
    epochs,
    batch_size,
    optimizer,
    lr,
    weight_decay,
    scheduler,
    num_classes,
    device,
    weight_init=None,
    momentum=0.9,
    betas=(0.9, 0.999),
    warmup_fraction=0.0,
    label_smoothing=0.0,
    l2sp=0.0,
    grad_clip=None,
    save_best_val_checkpoint=False,
    generator=None,
    desc='linear head',
):
    """Linear head on fixed features: cross-entropy + L2SP * ||W - W_init||^2. Returns (weight, bias, history) on CPU"""
    x = torch.as_tensor(train_features, dtype=torch.float32).to(device)
    y = torch.as_tensor(train_labels, dtype=torch.long).to(device)
    head = nn.Linear(x.shape[1], num_classes).to(device)

    if weight_init is not None:
        with torch.no_grad():
            head.weight.copy_(torch.as_tensor(weight_init, dtype=torch.float32))
            head.bias.zero_()

    weight_0 = head.weight.detach().clone()
    steps_per_epoch = math.ceil(len(x) / batch_size)
    opt = make_optimizer(
        [
            {'params': [head.weight], 'lr': lr, 'weight_decay': weight_decay, 'name': 'head'},
            {'params': [head.bias], 'lr': lr, 'weight_decay': 0.0, 'name': 'head.bias'},
        ],
        optimizer,
        lr=lr,
        weight_decay=weight_decay,
        momentum=momentum,
        betas=betas,
    )
    sched = make_scheduler(opt, scheduler, epochs * steps_per_epoch, warmup_fraction)
    best_accuracy, best_state, history = -1.0, None, []

    for epoch in tqdm(range(1, epochs + 1), desc=desc, unit='epoch', dynamic_ncols=True):
        total_loss, correct = 0.0, 0

        for index in torch.randperm(len(x), generator=generator).to(device).split(batch_size):
            logits = head(x[index])
            loss = F.cross_entropy(logits, y[index], label_smoothing=label_smoothing)

            if l2sp:
                loss = loss + l2sp * ((head.weight - weight_0) ** 2).sum()

            opt.zero_grad(set_to_none=True)
            loss.backward()

            if grad_clip:
                torch.nn.utils.clip_grad_norm_(head.parameters(), grad_clip)

            opt.step()
            sched.step()
            total_loss += loss.item() * len(index)
            correct += (logits.argmax(1) == y[index]).sum().item()

        val_logits = linear_logits(val_features, head.weight.detach(), head.bias.detach(), device=device)
        val_acc = accuracy(val_logits, val_labels)
        drift = float((head.weight.detach() - weight_0).norm() / weight_0.norm().clamp_min(1e-8))
        history.append(
            {
                'epoch': epoch,
                'loss': total_loss / len(x),
                'train_acc': correct / len(x),
                'val_acc': val_acc,
                'drift': drift,
            }
        )

        if save_best_val_checkpoint and val_acc > best_accuracy:
            best_accuracy, best_state = val_acc, {k: v.detach().clone() for k, v in head.state_dict().items()}

    log.info('[%s] %s', desc, history[-1])

    if best_state is not None:
        head.load_state_dict(best_state)
        log.info('[%s] restored the best CIFAR-10 validation epoch (val_acc=%.4f)', desc, best_accuracy)

    return head.weight.detach().cpu(), head.bias.detach().cpu(), pd.DataFrame(history)

### Logging and data

In [6]:
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S', force=True
)
log = logging.getLogger('recvis')

train_images, train_labels, train_ids = load_npz(CFG.input_dir / 'train.npz')
val_images, val_labels, val_ids = load_npz(CFG.input_dir / 'validation.npz')
test_images, test_labels, test_ids = load_npz(CFG.input_dir / 'test.npz')
assert test_labels is None, 'Test labels must never be used'
sample_submission = pd.read_csv(CFG.input_dir / 'sample_submission.csv')

if not np.array_equal(sample_submission['id'].to_numpy(), test_ids):
    raise ValueError('The ids of sample_submission.csv are not aligned with test.npz')

log.info('train %s | validation %s | test %s | device %s', train_images.shape, val_images.shape, test_images.shape, DEVICE)

01:13:01 | INFO | train (50000, 32, 32, 3) | validation (10000, 32, 32, 3) | test (51077, 32, 32, 3) | device cpu


## 1. ResNet / ViT

`trainable_blocks`: `0` linear probe, `k` head + last `k` blocks, `-1` full fine-tuning (linear probe and partial
fine-tuning need pretrained weights). Blocks are residual blocks (ResNet) or transformer blocks (ViT); the stem / patch
embedding is the first unit. No augmentation unless listed in `augmentations`.

| | head | trainable backbone blocks |
|---|---|---|
| lr | `lr` | ResNet: `backbone_lr` (usually `lr * 0.1`); ViT: `backbone_lr * layer_decay ** (d - 1)` at depth `d` from the top (usually `backbone_lr = lr * layer_decay`) |
| weight decay | `weight_decay` | `backbone_weight_decay`; never on biases, norms, class token, position embedding |
| L2-SP | `l2sp` (towards the initial head) | `backbone_l2sp` (towards the pretrained weights) |

A frozen backbone without augmentation caches the features once and trains only the head.

### Constant values

In [7]:
AUGMENTATIONS = ('crop', 'flip', 'color', 'grayscale', 'augmix', 'randaugment')

### Configuration and helpers

In [8]:
@dataclass
class BackboneConfig:
    # Model
    library: str = 'timm'  # Either 'torchvision' or 'timm'
    model_name: str = 'vit_huge_patch14_224.orig_in21k'
    pretrained: bool = True
    weights: str = 'DEFAULT'  # Only for torchvision; timm takes the weights from model_name
    trainable_blocks: int = 0  # Trainable units: 0 linear probe, k last k blocks, -1 all
    freeze_bn_stats: bool = True
    image_size: int = 224  # 32 for a ResNet from scratch
    resize_interpolation: str = 'bicubic'
    
    # Augmentation (train, 32px)
    augmentations: tuple = ()
    crop: str = 'random_resized_crop'  # 'random_crop' for a ResNet from scratch
    crop_padding: int = 4
    crop_scale: tuple = (0.8, 1.0)
    hflip_p: float = 0.5
    
    # Head
    normalize_features: bool = True
    head_batch_size: int = 1024
    l2sp: float = 1e-4  # L2SP * ||W_head - W_head_init||^2
    
    # Optimization
    batch_size: int = 128
    epochs: int = 20
    optimizer: str = 'adamw'
    lr: float = 1e-3            
    backbone_lr: float = 7.5e-4  # ViT: LR of the last block, LR * layer_decay; ResNet: usually LR * 0.1
    layer_decay: float = 0.75
    weight_decay: float = 0.0
    backbone_weight_decay: float = 0.0
    backbone_l2sp: float = 0.0  # Towards the pretrained weights
    momentum: float = 0.9
    betas: tuple = (0.9, 0.999)
    scheduler: str = 'cosine'
    warmup_fraction: float = 0.0
    label_smoothing: float = 0.0
    grad_clip: float = None
    save_best_val_checkpoint: bool = True
    tta_flip: bool = True  # Cached features: every split, else validation and test
    features_from: str = None  # Experiment folder: reuse its cache/ features
    seed: int = CFG.seed

    def __post_init__(self):
        if self.trainable_blocks < -1:
            raise ValueError('The trainable_blocks value must be -1, 0 or a positive number of blocks')
        if not self.pretrained and self.trainable_blocks != -1:
            raise ValueError('Linear probe and partial fine-tuning need pretrained weights (else trainable_blocks=-1)')
        if not self.pretrained and self.backbone_l2sp:
            raise ValueError('The backbone_l2sp option needs pretrained weights')

        self.augmentations = tuple(self.augmentations)

        if {'augmix', 'randaugment'} <= set(self.augmentations):
            raise ValueError('Use augmix or randaugment, not both')

        if self.features_from and not self.cached_features:
            raise ValueError('The features_from option needs a linear probe without augmentation')

    @property
    def cached_features(self):
        """Frozen backbone, no augmentation: features are computed once and only the head is trained"""
        return self.trainable_blocks == 0 and not self.augmentations

    @property
    def is_vit(self):
        return 'vit' in self.model_name.lower()

    @property
    def name(self):
        mode = {0: 'lp', -1: 'ft'}.get(self.trainable_blocks, f'ft{self.trainable_blocks}')
        aug = '_aug' if self.augmentations else ''

        return safe_tag(f"{self.model_name}{'_pt' if self.pretrained else ''}_{mode}{aug}")

    def feature_file(self, split):
        return safe_tag('feat', self.name, self.resize_interpolation, f'tta{int(self.tta_flip)}', split)

class BackboneClassifier(nn.Module):
    def __init__(self, backbone, feature_dim, normalize, num_classes):
        super().__init__()
        self.backbone, self.normalize = backbone, normalize
        self.head = nn.Linear(feature_dim, num_classes)

    def features(self, x):
        features = self.backbone(x)
        return F.normalize(features.float(), dim=-1) if self.normalize else features

    def forward(self, x):
        return self.head(self.features(x))

def build_backbone(cfg, scratch_mean, scratch_std):
    """Backbone without its classification head. Returns (backbone, feature_dim, info with mean/std/weights)

    scratch_mean / scratch_std: input normalization of a randomly initialized model
    """
    if cfg.library == 'torchvision':
        weights = models.get_model_weights(cfg.model_name)[cfg.weights] if cfg.pretrained else None
        backbone = models.get_model(cfg.model_name, weights=weights)

        if weights is not None:
            preprocess = weights.transforms()
            native, mean, std = int(preprocess.crop_size[0]), tuple(preprocess.mean), tuple(preprocess.std)
        else:
            native, mean, std = int(getattr(backbone, 'image_size', 224)), scratch_mean, scratch_std

        if hasattr(backbone, 'fc'):
            feature_dim = backbone.fc.in_features
            backbone.fc = nn.Identity()
        elif hasattr(backbone, 'heads'):
            feature_dim = backbone.hidden_dim
            backbone.heads = nn.Identity()
        else:
            raise ValueError(f'Model {cfg.model_name}: only torchvision ResNets and ViTs are supported')

        weights_name = str(weights) if weights is not None else 'random init'
    elif cfg.library == 'timm':
        options = {'img_size': cfg.image_size} if cfg.image_size and cfg.is_vit else {}
        backbone = timm.create_model(cfg.model_name, pretrained=cfg.pretrained, num_classes=0, **options)
        pretrained_cfg = backbone.pretrained_cfg
        native = int(cfg.image_size or pretrained_cfg['input_size'][-1])

        if cfg.pretrained:
            mean, std = tuple(pretrained_cfg['mean']), tuple(pretrained_cfg['std'])
            weights_name = f"timm {pretrained_cfg.get('hf_hub_id') or cfg.model_name}"
        else:
            mean, std, weights_name = scratch_mean, scratch_std, 'random init'

        feature_dim = backbone.num_features
    else:
        raise ValueError(f"Library must be 'torchvision' or 'timm', got {cfg.library}")

    if cfg.library == 'torchvision' and cfg.is_vit and cfg.image_size != native:
        raise ValueError(f'Model {cfg.model_name} requires image_size={native}')

    modifications = ['classification head removed']

    if not cfg.pretrained and cfg.image_size == 32 and isinstance(getattr(backbone, 'conv1', None), nn.Conv2d):
        backbone.conv1 = nn.Conv2d(3, backbone.conv1.out_channels, 3, 1, 1, bias=False)
        backbone.maxpool = nn.Identity()
        modifications.append('CIFAR stem: conv1 3x3 stride 1, no maxpool')

    info = {
        'library': cfg.library,
        'architecture': cfg.model_name,
        'class': type(backbone).__name__,
        'weights': weights_name,
        'input_size': cfg.image_size,
        'feature_dim': int(feature_dim),
        'mean': mean,
        'std': std,
        'parameters': sum(p.numel() for p in backbone.parameters()),
        'modifications': modifications,
    }

    return backbone, int(feature_dim), info

def backbone_units(backbone):
    """Units [(name, modules, parameters)] from the input to the output: stem/embedding, block 1, ..., block n (+ final norm)"""
    
    if hasattr(backbone, 'layer4'):  # ResNet (torchvision or timm)
        units = [('stem', [backbone.conv1, backbone.bn1])]
        
        units += [
            (f'{stage}.{i}', [block])
            for stage in ('layer1', 'layer2', 'layer3', 'layer4')
            for i, block in enumerate(getattr(backbone, stage))
        ]
    elif hasattr(backbone, 'blocks'):  # ViT from timm
        units = [
            ('embedding', [m for m in (backbone.patch_embed, getattr(backbone, 'norm_pre', None)) if m is not None])
        ]
        
        units += [(f'blocks.{i}', [block]) for i, block in enumerate(backbone.blocks)]
        
        units[-1][1].extend(
            m for m in (getattr(backbone, 'norm', None), getattr(backbone, 'fc_norm', None)) if m is not None
        )
    elif hasattr(backbone, 'encoder'):  # ViT from torchvision
        
        units = [('embedding', [backbone.conv_proj])]
        units += [(f'encoder.layers.{i}', [block]) for i, block in enumerate(backbone.encoder.layers)]
        units[-1][1].append(backbone.encoder.ln)
    else:
        raise ValueError(f'Unknown block structure for {type(backbone).__name__}')

    units = [(name, modules, [p for m in modules for p in m.parameters()]) for name, modules in units]
    covered = {id(p) for _, _, params in units for p in params}

    for name, p in backbone.named_parameters():  # Class token / position embedding -> embedding, the rest -> last unit
        if id(p) not in covered:
            embedding = any(
                key in name for key in ('cls_token', 'class_token', 'pos_embed', 'pos_embedding', 'reg_token')
            )
            units[0 if embedding else -1][2].append(p)

    return units

def configure_trainable(model, cfg):
    """Freeze all but the last trainable_blocks units. Returns (optimizer groups, frozen modules, info) """
    
    units = backbone_units(model.backbone)
    names = {id(p): name for name, p in model.backbone.named_parameters()}
    k = len(units) if cfg.trainable_blocks == -1 else min(cfg.trainable_blocks, len(units))

    for p in model.backbone.parameters():
        p.requires_grad = False

    groups = [
        {'params': [model.head.weight], 'lr': cfg.lr, 'weight_decay': cfg.weight_decay, 'name': 'head'},
        {'params': [model.head.bias], 'lr': cfg.lr, 'weight_decay': 0.0, 'name': 'head.bias'},
    ]
    frozen_modules, trainable_units = [], []

    for depth, (name, modules, params) in enumerate(reversed(units), start=1):
        if depth > k:
            frozen_modules += modules
            continue

        for p in params:
            p.requires_grad = True

        lr = cfg.backbone_lr * (cfg.layer_decay ** (depth - 1) if cfg.is_vit and cfg.pretrained else 1.0)
        no_decay = {id(p) for p in params if p.ndim <= 1 or any(t in names[id(p)] for t in ('token', 'pos_embed'))}
        decay = [p for p in params if id(p) not in no_decay]
        groups.append({'params': decay, 'lr': lr, 'weight_decay': cfg.backbone_weight_decay, 'l2sp': True, 'name': name})
        groups.append(
            {'params': [p for p in params if id(p) in no_decay], 'lr': lr, 'weight_decay': 0.0, 'name': f'{name}.no_decay'}
        )
        trainable_units.append(name)

    groups = [g for g in groups if g['params']]
    info = {
        'units': len(units),
        'trainable_units': trainable_units[::-1],
        'lr': {g['name']: g['lr'] for g in groups if not g['name'].endswith(('bias', 'no_decay'))},
        'weight_decay': {'head': cfg.weight_decay, 'backbone': cfg.backbone_weight_decay, 'biases_norms_tokens': 0.0},
        'l2sp': {'head': cfg.l2sp, 'backbone': cfg.backbone_l2sp},
        'trainable_parameters': sum(p.numel() for g in groups for p in g['params']),
    }
    log.info(
        '%s: %d/%d units trainable, %.2fM trainable parameters',
        cfg.name,
        k,
        len(units),
        info['trainable_parameters'] / 1e6,
    )

    return groups, frozen_modules, info

def build_augmentation(cfg, available):
    """Training transform at 32px: the augmentations of cfg, in the order of available"""
    unknown = set(cfg.augmentations) - set(available)

    if unknown:
        raise ValueError(f'Unknown augmentations {sorted(unknown)}; choose from {available}')

    ops = {
        'crop': (
            T.RandomCrop(32, padding=cfg.crop_padding)
            if cfg.crop == 'random_crop'
            else T.RandomResizedCrop(32, scale=tuple(cfg.crop_scale))
        ),
        'flip': T.RandomHorizontalFlip(p=cfg.hflip_p),
        'color': T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        'grayscale': T.RandomGrayscale(p=0.2),
        'augmix': T.AugMix(severity=3, mixture_width=3),
        'randaugment': T.RandAugment(num_ops=2, magnitude=9),
    }

    return T.Compose([ops[name] for name in available if name in cfg.augmentations])

class ImageDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images, self.labels, self.transform = images, labels, transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.transform(Image.fromarray(self.images[index]))
        return torch.from_numpy(np.array(image, dtype=np.uint8)), int(self.labels[index])

def predict(model, images, cfg, mean, std, tta_flip, desc, device, amp):
    """Features and logits of a BackboneClassifier, one image at a time (batched)"""
    model.eval()
    features = extract_features(
        model.features,
        images,
        size=cfg.image_size,
        mean=mean,
        std=std,
        interpolation=cfg.resize_interpolation,
        tta_flip=tta_flip,
        normalize=cfg.normalize_features,
        device=device,
        amp=amp,
        desc=desc,
    )

    return features, linear_logits(features, model.head.weight.detach(), model.head.bias.detach(), device=device)

def train_model(
    model,
    groups,
    frozen_modules,
    cfg,
    *,
    mean,
    std,
    transform,
    train_images,
    train_labels,
    val_images,
    val_labels,
    generator,
    worker_init_fn,
    device,
    amp,
    use_grad_scaler,
    num_workers,
):
    """Head + trainable blocks on augmented images; frozen blocks in eval mode. Returns the history"""
    loader = DataLoader(
        ImageDataset(train_images, train_labels, transform),
        batch_size=cfg.batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=device.type == 'cuda',
        persistent_workers=num_workers > 0,
        generator=generator,
        worker_init_fn=worker_init_fn,
    )
    parameters = [p for g in groups for p in g['params']]
    trainable = {id(p) for p in parameters}
    keep = {k for k, p in model.named_parameters() if id(p) in trainable} | {k for k, _ in model.named_buffers()}
    head_weight_0 = model.head.weight.detach().clone()
    backbone_anchors = [(p, p.detach().clone()) for g in groups if g.get('l2sp') for p in g['params']]
    opt = make_optimizer(
        groups, cfg.optimizer, lr=cfg.lr, weight_decay=cfg.weight_decay, momentum=cfg.momentum, betas=cfg.betas
    )
    sched = make_scheduler(opt, cfg.scheduler, cfg.epochs * len(loader), cfg.warmup_fraction)
    scaler = torch.amp.GradScaler('cuda', enabled=use_grad_scaler)
    best_accuracy, best_state, history = -1.0, None, []

    for epoch in range(1, cfg.epochs + 1):
        model.train()

        if cfg.freeze_bn_stats:
            for module in frozen_modules:
                module.eval()

        start, total_loss, correct, seen = time.time(), 0.0, 0, 0
        progress = tqdm(loader, desc=f'{cfg.name} epoch {epoch}/{cfg.epochs}', unit='batch', dynamic_ncols=True)

        for images, labels in progress:
            inputs = to_model_input(images, cfg.image_size, mean, std, cfg.resize_interpolation, device)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(**amp):
                logits = model(inputs)
                loss = F.cross_entropy(logits, labels, label_smoothing=cfg.label_smoothing)

            if cfg.l2sp:
                loss = loss + cfg.l2sp * ((model.head.weight - head_weight_0) ** 2).sum()
            if cfg.backbone_l2sp:
                loss = loss + cfg.backbone_l2sp * sum(((p - p0) ** 2).sum() for p, p0 in backbone_anchors)

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()

            if cfg.grad_clip:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(parameters, cfg.grad_clip)

            scaler.step(opt)
            scaler.update()
            sched.step()
            seen += len(labels)
            total_loss += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            progress.set_postfix(loss=f'{total_loss / seen:.3f}', acc=f'{correct / seen:.3f}')

        val_features, val_logits = predict(model, val_images, cfg, mean, std, False, 'validation', device, amp)
        row = {
            'epoch': epoch,
            'loss': total_loss / seen,
            'train_acc': correct / seen,
            'val_acc': accuracy(val_logits, val_labels),
            'seconds': time.time() - start,
        }
        history.append(row)
        log.info('[%s] %s', cfg.name, row)

        if cfg.save_best_val_checkpoint and row['val_acc'] > best_accuracy:
            best_accuracy = row['val_acc']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep}

    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
        log.info('[%s] restored the best CIFAR-10 validation epoch (val_acc=%.4f)', cfg.name, best_accuracy)

    return pd.DataFrame(history)

### Setup training

In [9]:
bb_cfg = BackboneConfig(model_name='vit_base_patch16_224.orig_in21k')
bb_run = Run('backbone', bb_cfg, CFG, environment_info(DEVICE, AMP))
bb_transform = build_augmentation(bb_cfg, AUGMENTATIONS)
bb_backbone, bb_feature_dim, bb_info = build_backbone(bb_cfg, CFG.cifar_mean, CFG.cifar_std)
bb_model = BackboneClassifier(bb_backbone, bb_feature_dim, bb_cfg.normalize_features, CFG.num_classes).to(DEVICE)
bb_groups, bb_frozen_modules, bb_trainable = configure_trainable(bb_model, bb_cfg)
bb_mean, bb_std = bb_info['mean'], bb_info['std']
bb_run.record('model', {**bb_info, 'trainable': bb_trainable})

01:13:06 | INFO | run folder c:\michele\mva\courses\computer_vision\tp\tp3\recvis26_a3\output\results\20261123_0113 | {'library': 'timm', 'model_name': 'vit_base_patch16_224.orig_in21k', 'pretrained': True, 'weights': 'DEFAULT', 'trainable_blocks': 0, 'freeze_bn_stats': True, 'image_size': 224, 'resize_interpolation': 'bicubic', 'augmentations': [], 'crop': 'random_resized_crop', 'crop_padding': 4, 'crop_scale': [0.8, 1.0], 'hflip_p': 0.5, 'normalize_features': True, 'head_batch_size': 1024, 'l2sp': 0.0001, 'batch_size': 128, 'epochs': 20, 'optimizer': 'adamw', 'lr': 0.001, 'backbone_lr': 0.00075, 'layer_decay': 0.75, 'weight_decay': 0.0, 'backbone_weight_decay': 0.0, 'backbone_l2sp': 0.0, 'momentum': 0.9, 'betas': [0.9, 0.999], 'scheduler': 'cosine', 'warmup_fraction': 0.0, 'label_smoothing': 0.0, 'grad_clip': None, 'save_best_val_checkpoint': True, 'tta_flip': True, 'features_from': None, 'seed': 42}


01:13:08 | INFO | Loading pretrained weights from Hugging Face hub (timm/vit_base_patch16_224.orig_in21k)
01:13:08 | INFO | HTTP Request: HEAD https://huggingface.co/timm/vit_base_patch16_224.orig_in21k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
01:13:08 | INFO | [timm/vit_base_patch16_224.orig_in21k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
01:13:08 | INFO | vit_base_patch16_224.orig_in21k_pt_lp: 0/13 units trainable, 0.01M trainable parameters


### Training and validation

In [10]:
if bb_cfg.cached_features:
    bb_model.eval()
    bb_cached = {}

    for split, images in (('train', train_images), ('validation', val_images)):
        features = load_features(bb_cfg.features_from, bb_cfg.feature_file(split), len(images))

        if features is None:
            features = extract_features(
                bb_model.features,
                images,
                size=bb_cfg.image_size,
                mean=bb_mean,
                std=bb_std,
                interpolation=bb_cfg.resize_interpolation,
                tta_flip=bb_cfg.tta_flip,
                normalize=bb_cfg.normalize_features,
                device=DEVICE,
                amp=AMP,
                desc=f'{split} features',
            )
            bb_run.save_features(bb_cfg.feature_file(split), features)

        bb_cached[split] = features

    bb_train_features, bb_val_features = bb_cached['train'], bb_cached['validation']
    weight, bias, bb_history = train_head(
        bb_train_features,
        train_labels,
        bb_val_features,
        val_labels,
        epochs=bb_cfg.epochs,
        batch_size=bb_cfg.head_batch_size,
        optimizer=bb_cfg.optimizer,
        lr=bb_cfg.lr,
        weight_decay=bb_cfg.weight_decay,
        scheduler=bb_cfg.scheduler,
        num_classes=CFG.num_classes,
        device=DEVICE,
        momentum=bb_cfg.momentum,
        betas=bb_cfg.betas,
        warmup_fraction=bb_cfg.warmup_fraction,
        label_smoothing=bb_cfg.label_smoothing,
        l2sp=bb_cfg.l2sp,
        grad_clip=bb_cfg.grad_clip,
        save_best_val_checkpoint=bb_cfg.save_best_val_checkpoint,
        generator=bb_run.generator,
        desc=bb_cfg.name,
    )

    with torch.no_grad():
        bb_model.head.weight.copy_(weight)
        bb_model.head.bias.copy_(bias)

    bb_val_logits = linear_logits(bb_val_features, weight, bias, device=DEVICE)
else:
    bb_history = train_model(
        bb_model,
        bb_groups,
        bb_frozen_modules,
        bb_cfg,
        mean=bb_mean,
        std=bb_std,
        transform=bb_transform,
        train_images=train_images,
        train_labels=train_labels,
        val_images=val_images,
        val_labels=val_labels,
        generator=bb_run.generator,
        worker_init_fn=bb_run.worker_init_fn,
        device=DEVICE,
        amp=AMP,
        use_grad_scaler=USE_GRAD_SCALER,
        num_workers=CFG.num_workers,
    )
    bb_val_features, bb_val_logits = predict(
        bb_model, val_images, bb_cfg, bb_mean, bb_std, bb_cfg.tta_flip, 'validation', DEVICE, AMP
    )
    bb_run.save_features(bb_cfg.feature_file('validation'), bb_val_features)

bb_run.save_table('history', bb_history)
bb_run.record('cifar_val_acc', accuracy(bb_val_logits, val_labels))
bb_trainable_ids = {id(p) for g in bb_groups for p in g['params']}
bb_keep = {k for k, p in bb_model.named_parameters() if id(p) in bb_trainable_ids} | {
    k for k, _ in bb_model.named_buffers()
}
bb_run.save_checkpoint(
    bb_cfg.name,
    {
        'config': to_plain(bb_cfg),
        'model': bb_info,
        'trainable': bb_trainable,
        'state_dict': {k: v.cpu() for k, v in bb_model.state_dict().items() if k in bb_keep},
    },
)
bb_history

train features:   0%|          | 0/196 [00:00<?, ?batch/s]

validation features:   0%|          | 0/40 [00:00<?, ?batch/s]

vit_base_patch16_224.orig_in21k_pt_lp:   0%|          | 0/20 [00:00<?, ?epoch/s]

C:\Users\Michele\AppData\Local\Temp\ipykernel_20008\1141903651.py:199: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  drift = float((head.weight - weight_0).norm() / weight_0.norm().clamp_min(1e-8))
10:09:46 | INFO | [vit_base_patch16_224.orig_in21k_pt_lp] {'epoch': 20, 'loss': 0.44042548789978025, 'train_acc': 0.96454, 'val_acc': 0.9598000049591064, 'drift': 13.441943168640137}
10:09:46 | INFO | [vit_base_patch16_224.orig_in21k_pt_lp] restored the best CIFAR-10 validation epoch (val_acc=0.9598)


,epoch,loss,train_acc,val_acc,drift
0,1,2.079627,0.84248,0.9524,2.070022
1,2,1.656848,0.95874,0.9563,3.992578
2,3,1.318433,0.96108,0.9570,5.702012
3,4,1.066306,0.96208,0.9567,7.169114
4,5,0.886703,0.96206,0.9577,8.401897
5,6,0.760680,0.96252,0.9575,9.425331
6,7,0.671828,0.96276,0.9580,10.271372
7,8,0.608338,0.96280,0.9582,10.969511
8,9,0.562208,0.96302,0.9585,11.543731
9,10,0.528241,0.96336,0.9590,12.012961


### Test

In [11]:
bb_test_features = load_features(bb_cfg.features_from, bb_cfg.feature_file('test'), len(test_images))

if bb_test_features is None:
    bb_test_features, bb_test_logits = predict(
        bb_model, test_images, bb_cfg, bb_mean, bb_std, bb_cfg.tta_flip, 'test', DEVICE, AMP
    )
    bb_run.save_features(bb_cfg.feature_file('test'), bb_test_features)
else:
    bb_test_logits = linear_logits(
        bb_test_features, bb_model.head.weight.detach(), bb_model.head.bias.detach(), device=DEVICE
    )

bb_run.save_submission(bb_cfg.name, bb_test_logits, bb_val_logits, val_labels, sample_submission)
bb_run.finish()

test:   0%|          | 0/200 [00:00<?, ?batch/s]

18:21:55 | INFO | saved submission_vit_base_patch16_224.orig_in21k_pt_lp.csv | {'name': 'vit_base_patch16_224.orig_in21k_pt_lp', 'cifar_val_acc': 0.9598000049591064, 'val_mean_max_prob': 0.7327, 'test_mean_max_prob': 0.4649, 'test_class_shares': array([0.1574, 0.0412, 0.1188, 0.1224, 0.0744, 0.0885, 0.0674, 0.0825,
       0.1072, 0.1402])}
18:21:55 | INFO | run finished: c:\michele\mva\courses\computer_vision\tp\tp3\recvis26_a3\output\results\20261123_0113 (9 files)


## 2. CLIP: zero-shot and linear probe + WiSE-FT

Zero-shot (`wise_alpha=None`): class embeddings = mean of the prompt embeddings. With `wise_alpha=a` a linear head
starts from the zero-shot classifier (`logit_scale * text embeddings`), is trained on the cached CIFAR-10 features with
L2-SP towards it, and is interpolated with it: `a = 0` zero-shot, `a = 1` trained head.

### Prompt templates

In [ ]:
# CLIP prompt templates (openai/CLIP: data/prompts.md for CIFAR-10, Prompt_Engineering_for_ImageNet.ipynb for ImageNet)
CIFAR_TEMPLATES = [
    'a photo of a {}.',
    'a blurry photo of a {}.',
    'a black and white photo of a {}.',
    'a low contrast photo of a {}.',
    'a high contrast photo of a {}.',
    'a bad photo of a {}.',
    'a good photo of a {}.',
    'a photo of a small {}.',
    'a photo of a big {}.',
    'a photo of the {}.',
    'a blurry photo of the {}.',
    'a black and white photo of the {}.',
    'a low contrast photo of the {}.',
    'a high contrast photo of the {}.',
    'a bad photo of the {}.',
    'a good photo of the {}.',
    'a photo of the small {}.',
    'a photo of the big {}.',
]

IMAGENET_TEMPLATES = [
    'a bad photo of a {}.',
    'a photo of many {}.',
    'a sculpture of a {}.',
    'a photo of the hard to see {}.',
    'a low resolution photo of the {}.',
    'a rendering of a {}.',
    'graffiti of a {}.',
    'a bad photo of the {}.',
    'a cropped photo of the {}.',
    'a tattoo of a {}.',
    'the embroidered {}.',
    'a photo of a hard to see {}.',
    'a bright photo of a {}.',
    'a photo of a clean {}.',
    'a photo of a dirty {}.',
    'a dark photo of the {}.',
    'a drawing of a {}.',
    'a photo of my {}.',
    'the plastic {}.',
    'a photo of the cool {}.',
    'a close-up photo of a {}.',
    'a black and white photo of the {}.',
    'a painting of the {}.',
    'a painting of a {}.',
    'a pixelated photo of the {}.',
    'a sculpture of the {}.',
    'a bright photo of the {}.',
    'a cropped photo of a {}.',
    'a plastic {}.',
    'a photo of the dirty {}.',
    'a jpeg corrupted photo of a {}.',
    'a blurry photo of the {}.',
    'a photo of the {}.',
    'a good photo of the {}.',
    'a rendering of the {}.',
    'a {} in a video game.',
    'a photo of one {}.',
    'a doodle of a {}.',
    'a close-up photo of the {}.',
    'a photo of a {}.',
    'the origami {}.',
    'the {} in a video game.',
    'a sketch of a {}.',
    'a doodle of the {}.',
    'a origami {}.',
    'a low resolution photo of a {}.',
    'the toy {}.',
    'a rendition of the {}.',
    'a photo of the clean {}.',
    'a photo of a large {}.',
    'a rendition of a {}.',
    'a photo of a nice {}.',
    'a photo of a weird {}.',
    'a blurry photo of a {}.',
    'a cartoon {}.',
    'art of a {}.',
    'a sketch of the {}.',
    'a embroidered {}.',
    'a pixelated photo of a {}.',
    'itap of the {}.',
    'a jpeg corrupted photo of the {}.',
    'a good photo of a {}.',
    'a plushie {}.',
    'a photo of the nice {}.',
    'a photo of the small {}.',
    'a photo of the weird {}.',
    'the cartoon {}.',
    'art of the {}.',
    'a drawing of the {}.',
    'a photo of the large {}.',
    'a black and white photo of a {}.',
    'the plushie {}.',
    'a dark photo of a {}.',
    'itap of a {}.',
    'graffiti of the {}.',
    'a toy {}.',
    'itap of my {}.',
    'a photo of a cool {}.',
    'a photo of a small {}.',
    'a tattoo of the {}.',
]

PROMPT_SETS = {
    'base': ['a photo of a {}.'],
    'cifar': CIFAR_TEMPLATES,
    'imagenet': IMAGENET_TEMPLATES,
    'cifar+imagenet': CIFAR_TEMPLATES + [t for t in IMAGENET_TEMPLATES if t not in CIFAR_TEMPLATES],
}

### Configuration and helpers

In [ ]:
@dataclass
class ClipConfig:
    clip_model: str = 'ViT-H-14-quickgelu'
    clip_pretrained: str = 'dfn5b'
    prompt_set: str = 'imagenet'
    interpolation: str = 'bicubic'
    tta_flip: bool = True
    batch_size: int = 128  # 256 in the WiSE experiments
    wise_alpha: float = None  # None: zero-shot; a: W = (1 - a) W_zeroshot + a W_trained, b = a b_trained
    epochs: int = 20
    batch_size_head: int = 1024
    optimizer: str = 'adamw'
    scheduler: str = 'cosine'
    warmup_fraction: float = 0.0
    lr: float = 1e-3
    weight_decay: float = 0.0
    l2sp: float = 1e-4  # Towards the zero-shot head
    label_smoothing: float = 0.0
    save_best_val_checkpoint: bool = False
    features_from: str = None  # Experiment folder: reuse its cache/ features
    seed: int = CFG.seed

    @property
    def tag(self):
        return safe_tag(self.clip_model, self.clip_pretrained)

    @property
    def prompt_tag(self):
        return self.prompt_set.replace('+', '-')

    @property
    def name(self):
        if self.wise_alpha is None:
            return f'clip_zeroshot_{self.tag}_prompt_{self.prompt_tag}'

        return f'clip_wise_a{round(self.wise_alpha * 100):03d}_{self.tag}_prompt_{self.prompt_tag}'

    def feature_file(self, split):
        return safe_tag('clipfeat', self.tag, split, self.interpolation, f'tta{int(self.tta_flip)}')

def load_clip(cfg, device):
    """OpenCLIP model on device + tokenizer, input size, normalization and logit scale"""
    name = cfg.clip_model

    if not open_clip.pretrained.get_pretrained_cfg(name, cfg.clip_pretrained) and name.endswith('-quickgelu'):
        name = name[: -len('-quickgelu')]  # OpenCLIP >= 3 registers DFN/MetaCLIP tags without the suffix

    model, _, preprocess = open_clip.create_model_and_transforms(
        name, pretrained=cfg.clip_pretrained, force_quick_gelu=cfg.clip_model.endswith('-quickgelu')
    )
    model = model.to(device).eval()
    normalize = next(t for t in preprocess.transforms if isinstance(t, T.Normalize))
    size = model.visual.image_size
    info = {
        'model': cfg.clip_model,
        'loaded_as': name,
        'pretrained': cfg.clip_pretrained,
        'input_size': size[0] if isinstance(size, (tuple, list)) else size,
        'mean': tuple(normalize.mean),
        'std': tuple(normalize.std),
        'logit_scale': model.logit_scale.exp().item(),
        'parameters': sum(p.numel() for p in model.parameters()),
    }

    return model, open_clip.get_tokenizer(name), info

@torch.no_grad()
def text_classifier(model, tokenizer, templates, class_names, device, amp):
    """Class embeddings (classes, d): mean of the L2-normalized prompt embeddings, L2-normalized"""
    weights = []

    for name in class_names:
        tokens = tokenizer([template.format(name) for template in templates]).to(device)

        with torch.autocast(**amp):
            embeddings = F.normalize(model.encode_text(tokens).float(), dim=-1)

        weights.append(F.normalize(embeddings.mean(0), dim=0))

    return torch.stack(weights).cpu()

### Run

In [ ]:
clip_cfg = ClipConfig()
clip_templates = PROMPT_SETS[clip_cfg.prompt_set]
clip_run = Run('clip', clip_cfg, CFG, environment_info(DEVICE, AMP))
clip_model, clip_tokenizer, clip_info = load_clip(clip_cfg, DEVICE)
clip_run.record('model', clip_info)
clip_text = text_classifier(clip_model, clip_tokenizer, clip_templates, CFG.class_names, DEVICE, AMP)
clip_run.record('prompts', {'set': clip_cfg.prompt_set, 'templates': clip_templates})

clip_splits = {'validation': val_images, 'test': test_images}

if clip_cfg.wise_alpha is not None:
    clip_splits['train'] = train_images

clip_features = {}

for split, images in clip_splits.items():
    features = load_features(clip_cfg.features_from, clip_cfg.feature_file(split), len(images))

    if features is None:
        features = extract_features(
            clip_model.encode_image,
            images,
            size=clip_info['input_size'],
            mean=clip_info['mean'],
            std=clip_info['std'],
            interpolation=clip_cfg.interpolation,
            tta_flip=clip_cfg.tta_flip,
            normalize=True,
            device=DEVICE,
            amp=AMP,
            batch_size=clip_cfg.batch_size,
            desc=f'CLIP {split}',
        )
        clip_run.save_features(clip_cfg.feature_file(split), features)

    clip_features[split] = features

clip_val_features = clip_features['validation']

clip_w0 = clip_info['logit_scale'] * clip_text
clip_zeroshot_val = linear_logits(clip_val_features, clip_w0, device=DEVICE)
clip_zeroshot_test = linear_logits(clip_features['test'], clip_w0, device=DEVICE)
clip_zeroshot_name = f'clip_zeroshot_{clip_cfg.tag}_prompt_{clip_cfg.prompt_tag}'
clip_run.save_submission(clip_zeroshot_name, clip_zeroshot_test, clip_zeroshot_val, val_labels, sample_submission)

if clip_cfg.wise_alpha is not None:
    clip_w1, clip_b1, clip_history = train_head(
        clip_features['train'],
        train_labels,
        clip_val_features,
        val_labels,
        weight_init=clip_w0,
        epochs=clip_cfg.epochs,
        batch_size=clip_cfg.batch_size_head,
        optimizer=clip_cfg.optimizer,
        lr=clip_cfg.lr,
        weight_decay=clip_cfg.weight_decay,
        scheduler=clip_cfg.scheduler,
        num_classes=CFG.num_classes,
        device=DEVICE,
        warmup_fraction=clip_cfg.warmup_fraction,
        label_smoothing=clip_cfg.label_smoothing,
        l2sp=clip_cfg.l2sp,
        save_best_val_checkpoint=clip_cfg.save_best_val_checkpoint,
        generator=clip_run.generator,
        desc='CLIP head',
    )
    clip_run.save_table('head_history', clip_history)
    clip_run.save_checkpoint(
        f'clip_heads_{clip_cfg.tag}_prompt_{clip_cfg.prompt_tag}',
        {
            'zeroshot_weight': clip_w0,
            'finetuned_weight': clip_w1,
            'finetuned_bias': clip_b1,
            'config': to_plain(clip_cfg),
            'model': clip_info,
        },
    )
    alpha = clip_cfg.wise_alpha
    clip_weight, clip_bias = (1 - alpha) * clip_w0 + alpha * clip_w1, alpha * clip_b1
    clip_val_logits = linear_logits(clip_val_features, clip_weight, clip_bias, device=DEVICE)
    clip_test_logits = linear_logits(clip_features['test'], clip_weight, clip_bias, device=DEVICE)
    clip_run.save_submission(clip_cfg.name, clip_test_logits, clip_val_logits, val_labels, sample_submission)
    clip_run.record(
        'wise',
        {
            'alpha': alpha,
            'cifar_val_acc': accuracy(clip_val_logits, val_labels),
            'test_agreement_with_zeroshot': float(
                (clip_test_logits.argmax(1) == clip_zeroshot_test.argmax(1)).float().mean()
            ),
            'test_mean_confidence': clip_test_logits.softmax(1).max(1).values.mean().item(),
        },
    )

clip_run.finish()